In [1]:
import os
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

MODEL_PATH = r"D:\Models\Qwen3-14B"
OFFLOAD_PATH = r"D:\python_environment\offload"

os.makedirs(OFFLOAD_PATH, exist_ok=True)

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    local_files_only=True,
)

quantization_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_enable_fp32_cpu_offload=True,
)

print("Loading model...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=quantization_config,
    device_map="auto",

    # Leave some VRAM available for Windows and text generation.
    max_memory={
        0: "6GiB",
        "cpu": "24GiB",
    },

    offload_folder=OFFLOAD_PATH,
    offload_state_dict=True,
    local_files_only=True,
    low_cpu_mem_usage=True,
    torch_dtype=torch.float16,
)

model.eval()

print("Model loaded successfully.")
print(model.hf_device_map)

PyTorch: 2.11.0+cu128
CUDA: 12.8
GPU: NVIDIA GeForce RTX 4060 Ti


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading model...


W0709 23:05:22.023000 28640 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Loading weights:   0%|          | 0/443 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


Model loaded successfully.
{'model.embed_tokens': 0, 'model.layers.0': 0, 'model.layers.1': 0, 'model.layers.2': 0, 'model.layers.3': 0, 'model.layers.4': 0, 'model.layers.5': 0, 'model.layers.6': 0, 'model.layers.7': 0, 'model.layers.8': 'cpu', 'model.layers.9': 'cpu', 'model.layers.10': 'cpu', 'model.layers.11': 'cpu', 'model.layers.12': 'cpu', 'model.layers.13': 'cpu', 'model.layers.14': 'cpu', 'model.layers.15': 'cpu', 'model.layers.16': 'cpu', 'model.layers.17': 'cpu', 'model.layers.18': 'cpu', 'model.layers.19': 'cpu', 'model.layers.20': 'cpu', 'model.layers.21': 'cpu', 'model.layers.22': 'cpu', 'model.layers.23': 'cpu', 'model.layers.24': 'cpu', 'model.layers.25': 'cpu', 'model.layers.26': 'cpu', 'model.layers.27': 'cpu', 'model.layers.28': 'cpu', 'model.layers.29': 'cpu', 'model.layers.30': 'cpu', 'model.layers.31': 'cpu', 'model.layers.32': 'cpu', 'model.layers.33': 'cpu', 'model.layers.34': 'cpu', 'model.layers.35': 'cpu', 'model.layers.36': 'cpu', 'model.layers.37': 'cpu', '

In [2]:
messages = [
    {
        "role": "system",
        "content": (
            "You are a helpful assistant. Give clear, accurate, "
            "and well-structured answers."
        ),
    },
    {
        "role": "user",
        "content": "Explain how a random forest model predicts wildfire risk.",
    },
]

formatted_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

inputs = tokenizer(
    formatted_prompt,
    return_tensors="pt",
)

# Put the prompt on the GPU where the first model layers run.
inputs = {
    name: tensor.to("cuda:0")
    for name, tensor in inputs.items()
}

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.7,
        top_p=0.8,
        repetition_penalty=1.05,
        pad_token_id=tokenizer.eos_token_id,
    )

new_tokens = outputs[0, inputs["input_ids"].shape[1]:]

response = tokenizer.decode(
    new_tokens,
    skip_special_tokens=True,
).strip()

print(response)

A **Random Forest** model is an ensemble learning method that uses multiple decision trees to make predictions. When applied to **wildfire risk prediction**, it leverages historical data on wildfires and environmental conditions to estimate the likelihood of a fire occurring in a given area. Here's a step-by-step explanation of how a Random Forest model predicts wildfire risk:

---

### 1. **Data Collection and Preparation**

To build a wildfire risk prediction model using Random Forest, you need **historical wildfire data** and **environmental variables**. This includes:

- **Historical fire records**: Locations, times, sizes, and causes of past wildfires.
- **Environmental features**:
  - Weather data (temperature, humidity, precipitation, wind speed)
  - Topography (elevation, slope, aspect)
  - Vegetation type and density
  - Fuel load (amount of dry vegetation available to burn)
  - Human activity (population density, proximity to roads or settlements)

These data are used to crea